# AI Code Auditor v4 — Evaluation (5 Classes)
**Model:** DeepSeek-Coder-6.7B + QLoRA v4 | **Test:** 5 CWE classes

**5 Target Classes:**
- CWE-20  : Improper Input Validation
- CWE-399 : Resource Management Errors
- CWE-264 : Permissions / Access Control
- CWE-190 : Integer Overflow (Synthetic Boosted)
- CWE-416 : Use After Free (Synthetic Boosted)

**Before running:**
1. GPU: T4 x1
2. Add dataset with: test.jsonl + lora_adapter_v4/ folder

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate==0.29.3 bitsandbytes==0.45.3 sacrebleu rouge-score
print('Done')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Environment set')

In [ ]:
import os, json, re
from collections import Counter

# 5 classes we are evaluating on
TARGET_5 = {'CWE-20', 'CWE-399', 'CWE-264', 'CWE-190', 'CWE-416'}
SAMPLES_PER_CLASS = 20  # 20 x 5 = 100 total, ~50 min on T4

# Find test set and adapter
TEST_PATH    = None
ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'test.jsonl':
            TEST_PATH = full
        if f == 'adapter_config.json' and 'checkpoint' not in root:
            ADAPTER_PATH = root

assert TEST_PATH,    'test.jsonl not found'
assert ADAPTER_PATH, 'adapter_config.json not found'
print('Test set :', TEST_PATH)
print('Adapter  :', ADAPTER_PATH)

# Load test records — 10 per class (50 total)
import random
random.seed(42)
with open(TEST_PATH) as f:
    all_records = [json.loads(l) for l in f]

# Sample exactly SAMPLES_PER_CLASS from each CWE
test_records = []
for cwe in TARGET_5:
    pool = [r for r in all_records if r['cwe'] == cwe]
    sampled = random.sample(pool, min(SAMPLES_PER_CLASS, len(pool)))
    test_records.extend(sampled)
random.shuffle(test_records)

print('Total test samples:', len(test_records), '(', SAMPLES_PER_CLASS, 'per class )')
print('Estimated time    : ~', len(test_records) * 30 // 60, 'minutes')
dist = Counter(r['cwe'] for r in test_records)
print('Distribution:')
for cwe, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(' ', cwe, ':', count, 'samples')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-base'
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
base_model.config.use_cache = True
print('Base model loaded. VRAM:', round(torch.cuda.memory_allocated()/1e9, 1), 'GB')

In [ ]:
# Load v4 fine-tuned adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print('v4 adapter loaded from:', ADAPTER_PATH)

In [ ]:
def build_prompt(code):
    return (
        'You are an expert security code auditor.\n'
        'Analyze the following C/C++ code and identify the security vulnerability.\n\n'
        '```c\n' + code + '\n```\n\n'
        'Respond with the CWE type first, then explain and provide a secure rewrite.\n'
        'CWE:'
    )

def extract_cwe(text):
    m = re.search(r'CWE-\d+', text)
    return m.group(0) if m else 'Unknown'

def run_inference(code, max_new_tokens=250):
    prompt = build_prompt(code)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=400).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Sanity check
test_out = run_inference('void foo(char *s) { char buf[8]; strcpy(buf, s); }')
print('Sanity check output:', test_out[:100])
print('Extracted CWE:', extract_cwe(test_out))

In [ ]:
from tqdm import tqdm

print('Running v4 inference on', len(test_records), 'samples (5 classes)...')
results = []
for i, record in enumerate(tqdm(test_records)):
    output = run_inference(record['vulnerable_code'])
    results.append({
        'sample_id'          : i,
        'ground_truth_cwe'   : record['cwe'],
        'predicted_cwe'      : extract_cwe(output),
        'ground_truth_secure': record['secure_code'],
        'predicted_secure'   : output,
        'raw_output'         : output,
        'vulnerable_code'    : record['vulnerable_code'],
    })
    # Auto-save every 10 samples
    if (i + 1) % 10 == 0:
        with open('/kaggle/working/finetuned_results_v4.jsonl', 'w') as f:
            for r in results:
                f.write(json.dumps(r) + '\n')
        done = sum(1 for r in results if r['ground_truth_cwe'] == r['predicted_cwe'])
        print('Checkpoint', i+1, '— Accuracy so far:', done, '/', len(results))

# Final save
with open('/kaggle/working/finetuned_results_v4.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

correct = sum(1 for r in results if r['ground_truth_cwe'] == r['predicted_cwe'])
unknown = sum(1 for r in results if r['predicted_cwe'] == 'Unknown')
print()
print('v4 CWE Accuracy (5 classes):', correct, '/', len(results), '=', round(correct/len(results)*100, 1), '%')
print('Unknown predictions         :', unknown)

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
import numpy as np

refs  = [r['ground_truth_secure'] for r in results]
hyps  = [r['predicted_secure']    for r in results]
gt    = [r['ground_truth_cwe']    for r in results]
pred  = [r['predicted_cwe']       for r in results]

bleu   = sacrebleu.corpus_bleu(hyps, [refs]).score
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rougeL = np.mean([scorer.score(r, h)['rougeL'].fmeasure for r, h in zip(refs, hyps)])
cwe_acc = sum(1 for g, p in zip(gt, pred) if g == p) / len(gt)

# Per-CWE breakdown
per_cwe = {}
for cwe in TARGET_5:
    cwe_gt   = [g for g in gt if g == cwe]
    cwe_pred = [p for g, p in zip(gt, pred) if g == cwe]
    if cwe_gt:
        correct_cwe = sum(1 for g, p in zip(cwe_gt, cwe_pred) if g == p)
        per_cwe[cwe] = {
            'correct' : correct_cwe,
            'total'   : len(cwe_gt),
            'accuracy': round(correct_cwe / len(cwe_gt), 3)
        }

print('=' * 55)
print('  v4 EVALUATION RESULTS (5 Classes)')
print('=' * 55)
print('  BLEU-4       :', round(bleu, 2))
print('  ROUGE-L      :', round(rougeL, 3))
print('  CWE Accuracy :', str(round(cwe_acc*100, 1)) + '%  (' + str(sum(1 for g,p in zip(gt,pred) if g==p)) + '/' + str(len(gt)) + ')')
print('  Unknown preds:', sum(1 for p in pred if p == 'Unknown'))
print()
print('  Per-CWE Accuracy:')

# Historical comparison
v2_acc = {'CWE-20': 0.28, 'CWE-399': 0.43, 'CWE-264': 0.60, 'CWE-190': 0.0, 'CWE-416': 0.0}
v3_acc = {'CWE-20': 0.40, 'CWE-399': 0.14, 'CWE-264': 0.40, 'CWE-190': 0.50, 'CWE-416': 0.25}

for cwe in sorted(per_cwe, key=lambda x: -per_cwe[x]['total']):
    stats = per_cwe[cwe]
    v2 = v2_acc.get(cwe, 0)
    v3 = v3_acc.get(cwe, 0)
    v4 = stats['accuracy']
    trend = 'UP' if v4 > v2 else ('SAME' if v4 == v2 else 'DOWN')
    bar = '#' * stats['correct'] + '.' * (stats['total'] - stats['correct'])
    print('  ' + cwe + ': ' + str(stats['correct']) + '/' + str(stats['total'])
          + ' (' + str(round(v4*100)) + '%)'
          + '  v2=' + str(round(v2*100)) + '%'
          + '  v3=' + str(round(v3*100)) + '%'
          + '  [' + trend + ']')

print('=' * 55)

# Save metrics
metrics = {
    'model'      : 'v4 Fine-tuned (QLoRA DeepSeek-6.7B, 5-class)',
    'bleu4'      : round(bleu, 2),
    'rougeL'     : round(rougeL, 3),
    'cwe_accuracy': round(cwe_acc, 3),
    'unknown_count': sum(1 for p in pred if p == 'Unknown'),
    'correct'    : sum(1 for g, p in zip(gt, pred) if g == p),
    'total'      : len(gt),
    'per_cwe'    : per_cwe,
    'classes_evaluated': list(TARGET_5),
    'train_loss' : 0.3816,
    'eval_loss_best': 0.5251,
}
with open('/kaggle/working/evaluation_metrics_v4.json', 'w') as f:
    json.dump({'finetuned_v4': metrics}, f, indent=2)
print('Saved evaluation_metrics_v4.json')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

cwes   = sorted(per_cwe.keys())
v2_vals = [v2_acc.get(c, 0) for c in cwes]
v3_vals = [v3_acc.get(c, 0) for c in cwes]
v4_vals = [per_cwe[c]['accuracy'] for c in cwes]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Per-CWE comparison
x = np.arange(len(cwes))
w = 0.25
axes[0].bar(x - w, v2_vals, w, label='v2 (baseline)', color='steelblue', alpha=0.8)
axes[0].bar(x,     v3_vals, w, label='v3 (aggressive)', color='coral',    alpha=0.8)
axes[0].bar(x + w, v4_vals, w, label='v4 (gentle)',    color='seagreen',  alpha=0.8)
axes[0].set_title('Per-CWE Accuracy: v2 vs v3 vs v4', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cwes, rotation=15)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.0)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Overall accuracy trend
versions  = ['v1\n(zero-shot)', 'v2\n(fine-tuned)', 'v3\n(aggressive)', 'v4\n(gentle)']
# Overall on 5 classes
v1_5class = (0*4 + 7*25 + 6*14 + 3*5 + 0*4) / (4+25+14+5+4)  # approx
v2_5class = sum(v2_acc[c] * per_cwe[c]['total'] for c in cwes) / sum(per_cwe[c]['total'] for c in cwes)
v3_5class = sum(v3_acc[c] * per_cwe[c]['total'] for c in cwes) / sum(per_cwe[c]['total'] for c in cwes)
v4_5class = cwe_acc
accs = [v1_5class, v2_5class, v3_5class, v4_5class]
colors = ['gray', 'steelblue', 'coral', 'seagreen']
bars = axes[1].bar(versions, accs, color=colors, alpha=0.85, width=0.5)
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 str(round(acc*100, 1)) + '%', ha='center', fontweight='bold')
axes[1].set_title('Overall CWE Accuracy (5 Classes)', fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('AI Code Auditor: All Versions Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/v2_v3_v4_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved v2_v3_v4_comparison.png')

In [ ]:
from IPython.display import FileLink, display
print('Download your results:')
display(FileLink('evaluation_metrics_v4.json'))
display(FileLink('finetuned_results_v4.jsonl'))
display(FileLink('v2_v3_v4_comparison.png'))